# CoreGuard RUL Engine - Exploratory Data Analysis

This notebook explores the NASA C-MAPSS FD001 dataset to understand:
1. What the data looks like (shape, columns, types)
2. How engines degrade over time (sensor trends)
3. Which sensors carry useful signal vs. which are constant/noisy
4. How RUL labels are distributed
5. Correlations between sensors and RUL

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import load_train_data, download_cmapss
from src.data.preprocessor import add_rul_labels, cap_rul
from src.config import USEFUL_SENSORS, MAX_RUL, COLUMN_NAMES

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('Setup complete.')

## 1. Load Raw Data

In [ ]:
# Download dataset if needed, then load
download_cmapss()
df = load_train_data()

print(f'Shape: {df.shape}')
print(f'Engines: {df["unit_id"].nunique()}')
print(f'Columns: {list(df.columns)}')
df.head(10)

## 2. Basic Statistics

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print('No missing values found.')

print(f'\nData types:\n{df.dtypes.value_counts()}')
print(f'\nEngine lifecycle lengths (cycles):')
lifecycle = df.groupby('unit_id')['cycle'].max()
print(lifecycle.describe())

## 3. Engine Lifecycle Distribution

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 5))
lifecycle.hist(bins=30, ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Total Cycles Before Failure')
ax.set_ylabel('Number of Engines')
ax.set_title('Distribution of Engine Lifespans (FD001)')
ax.axvline(lifecycle.mean(), color='red', linestyle='--', label=f'Mean: {lifecycle.mean():.0f} cycles')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Sensor Variance Analysis (Which Sensors Are Useful?)

In [ ]:
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
sensor_variance = df[sensor_cols].var().sort_values(ascending=False)

fig, ax = plt.subplots(1, 1, figsize=(14, 5))
colors = ['steelblue' if f'sensor_{i}' in USEFUL_SENSORS else 'lightgray'
          for i, _ in enumerate(sensor_variance.index, 1)]
# Fix color mapping to match actual sensor names
colors = ['steelblue' if name in USEFUL_SENSORS else 'lightgray' for name in sensor_variance.index]
sensor_variance.plot(kind='bar', ax=ax, color=colors, edgecolor='black')
ax.set_ylabel('Variance')
ax.set_title('Sensor Variance (Blue = Kept, Gray = Dropped)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Sensors with near-zero variance (DROPPED):')
dropped = [s for s in sensor_cols if s not in USEFUL_SENSORS]
for s in dropped:
    print(f'  {s}: variance = {df[s].var():.6f}')

## 5. RUL Distribution

In [ ]:
df = add_rul_labels(df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before capping
df['rul'].hist(bins=50, ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('RUL Distribution (Before Capping)')
axes[0].set_xlabel('RUL (cycles)')
axes[0].set_ylabel('Count')

# After capping
df_capped = cap_rul(df.copy())
df_capped['rul'].hist(bins=50, ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title(f'RUL Distribution (Capped at {MAX_RUL})')
axes[1].set_xlabel('RUL (cycles)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 6. Sensor Degradation Over Time (Sample Engines)

In [ ]:
# Pick 4 sample engines to visualize
sample_engines = [1, 10, 50, 80]
plot_sensors = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_11']

fig, axes = plt.subplots(len(plot_sensors), 1, figsize=(14, 3 * len(plot_sensors)))

for idx, sensor in enumerate(plot_sensors):
    ax = axes[idx]
    for engine in sample_engines:
        engine_data = df[df['unit_id'] == engine]
        ax.plot(engine_data['cycle'], engine_data[sensor], label=f'Engine {engine}', alpha=0.7)
    ax.set_ylabel(sensor)
    ax.set_xlabel('Cycle')
    ax.set_title(f'{sensor} Over Time')
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

## 7. Sensor Correlation with RUL

In [ ]:
# Calculate correlation of each useful sensor with RUL
correlations = df_capped[USEFUL_SENSORS + ['rul']].corr()['rul'].drop('rul').sort_values()

fig, ax = plt.subplots(1, 1, figsize=(12, 6))
colors = ['coral' if v < 0 else 'steelblue' for v in correlations.values]
correlations.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
ax.set_xlabel('Correlation with RUL')
ax.set_title('Sensor Correlation with Remaining Useful Life')
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  Negative correlation = sensor value INCREASES as engine degrades (RUL decreases)')
print('  Positive correlation = sensor value DECREASES as engine degrades')

## 8. Summary

Key findings from EDA:
- Dataset has 100 engines with varying lifespans (128-362 cycles)
- 14 out of 21 sensors show meaningful variance (others are near-constant)
- RUL capping at 125 creates a focused degradation zone for training
- Several sensors show strong linear correlation with RUL
- Sensor degradation patterns are visible across all sample engines